# Step 13. Cluster both axes, modules not imposed

Reads `step12_panels.rds`. Writes `step13_clusters.rds`.

Two arms over the same panels:

- Arm 1, hierarchical, ward.D2 on `dist(method = "minkowski", p = 2)`, on both axes, cut
  at one level (2 groups) and one level down (4). p = 2 is identical to Euclidean; another
  order is a one-line change.
- Arm 2, k-means, both axes, k = 3, 4, 5.

**The modules are not imposed.** Fixing `column_split` to the module labels would make
the heatmap show blocks *by construction*, since weighted gene co-expression network analysis (WGCNA) already grouped those proteins, the picture
always returned a module and would prove nothing. Run free, the blocks either reappear or they do not, and
either outcome is information.

The `ARI_vs_wgcna` column measures exactly that: how much the free protein clustering agrees with
module labels it never saw. Adjusted Rand index, 0 is chance, 1 is identical (Hubert & Arabie 1985).

In [1]:
source("../src/paths.R")
options(stringsAsFactors = FALSE); set.seed(42)
P <- readRDS(art("step12_panels.rds"))
D <- P$D; MINK_P <- P$MINK_P; LINK <- P$LINK; SITES <- P$SITES
W <- lapply(SITES, function(s) readRDS(art("wgcna_%s.rds", s))); names(W) <- SITES

KS <- 3:8                      # both axes, both arms
HC_EXTRA <- 2                  # the hierarchical arm also keeps the 2-group cut

ARI <- function(a,b){ tab<-table(a,b); n<-sum(tab); ch2<-function(x) x*(x-1)/2
  idx<-sum(ch2(tab)); ai<-sum(ch2(rowSums(tab))); bj<-sum(ch2(colSums(tab)))
  e<-ai*bj/ch2(n); m<-(ai+bj)/2; (idx-e)/(m-e) }
jaccard <- function(a,b) length(intersect(a,b))/length(union(a,b))
dmink <- function(m) dist(m, method="minkowski", p=MINK_P)

# Both arms return rows/cols keyed BY k, so callers index by k and not by
# position -- an earlier version indexed by position and silently drew the
# wrong cut when the k range changed.
arm_hclust <- function(Z, ks){
  hr <- hclust(dmink(Z),    method=LINK)    # patients
  hc <- hclust(dmink(t(Z)), method=LINK)    # proteins
  nm <- as.character(ks)
  # factor() drops the names cutree() returns, and best_jac() needs them to know
  # which protein is in which cluster.
  fac <- function(v, pre, id) setNames(factor(paste0(pre, v)), id)
  list(hr=hr, hc=hc,
       rows=setNames(lapply(ks, function(k) fac(cutree(hr,k), "P", rownames(Z))), nm),
       cols=setNames(lapply(ks, function(k) fac(cutree(hc,k), "M", colnames(Z))), nm))
}
arm_kmeans <- function(Z, ks){
  nm <- as.character(ks)
  fac <- function(v, pre, id) setNames(factor(paste0(pre, v)), id)
  list(rows=setNames(lapply(ks, function(k){ set.seed(42)
         fac(kmeans(Z,   k, nstart=50)$cluster, "P", rownames(Z)) }), nm),
       cols=setNames(lapply(ks, function(k){ set.seed(42)
         fac(kmeans(t(Z),k, nstart=50)$cluster, "M", colnames(Z)) }), nm))
}

RES <- list(); tab <- NULL
for (nm in names(D)){
  d <- D[[nm]]
  if (!length(d$sel)) { cat(sprintf("%-6s %-7s : empty panel, nothing to cluster\n",
                                    d$cohort, d$cond)); next }
  Z <- scale(W[[d$cohort]]$X[, d$sel, drop=FALSE])
  ks_h <- sort(unique(c(HC_EXTRA, KS)))
  a1 <- arm_hclust(Z, ks_h); a2 <- arm_kmeans(Z, KS)
  mods <- W[[d$cohort]]$mods[match(d$sel, colnames(W[[d$cohort]]$X))]
  names(mods) <- d$sel
  RES[[nm]] <- list(Z=Z, hclust=a1, kmeans=a2, mods=mods, d=d, ks=KS, ks_h=ks_h)

  # best Jaccard of ANY protein cluster against the WGCNA module it never saw.
  # This is the statistic that showed a 10% panel cut destroyed the interferon
  # module: at 765 probes it never exceeds 0.14, at 164 it reaches 1.00.
  best_jac <- function(cl){
    grp <- split(names(cl), cl)
    sapply(setdiff(unique(mods), "grey"), function(m){
      ref <- names(mods)[mods==m]; max(sapply(grp, function(g) jaccard(g, ref))) })
  }
  addrow <- function(arm, k, rows, cols){
    bj <- best_jac(cols)
    data.frame(cohort=d$cohort, cond=d$cond, arm=arm, k=k,
      patient_sizes=paste(sort(table(rows)),collapse="/"),
      protein_sizes=paste(sort(table(cols)),collapse="/"),
      ARI_vs_wgcna=round(ARI(cols, mods),3),
      n_recovered=sum(bj >= 0.99),
      best_module=names(bj)[which.max(bj)], best_jaccard=round(max(bj),2)) }
  for (k in ks_h) tab <- rbind(tab, addrow("hclust", k, a1$rows[[as.character(k)]],
                                                        a1$cols[[as.character(k)]]))
  for (k in KS)   tab <- rbind(tab, addrow("kmeans", k, a2$rows[[as.character(k)]],
                                                        a2$cols[[as.character(k)]]))
}
cat("\nfree clustering on both axes; modules are NOT imposed.\n")
cat("ARI compares the PROTEIN clusters to the WGCNA labels; n_recovered counts\n")
cat("modules a free cluster reproduces exactly (Jaccard >= 0.99).\n\n")
print(tab[tab$cond=="all15",], row.names=FALSE)

saveRDS(list(RES = RES, tab = tab, KS = KS), art("step13_clusters.rds"))


B      all15   : empty panel, nothing to cluster
B      varsel  : empty panel, nothing to cluster
B      union   : empty panel, nothing to cluster



free clustering on both axes; modules are NOT imposed.


ARI compares the PROTEIN clusters to the WGCNA labels; n_recovered counts


modules a free cluster reproduces exactly (Jaccard >= 0.99).



 cohort  cond    arm k       patient_sizes           protein_sizes ARI_vs_wgcna
      A all15 hclust 2                2/85                   69/95        0.280
      A all15 hclust 3              2/9/76                36/59/69        0.476
      A all15 hclust 4            2/4/5/76             31/36/38/59        0.643
      A all15 hclust 5         2/4/5/23/53          14/22/31/38/59        0.698
      A all15 hclust 6      2/4/5/23/25/28       10/14/21/22/38/59        0.739
      A all15 hclust 7    2/4/5/9/19/23/25    10/10/14/21/22/38/49        0.846
      A all15 hclust 8  2/2/3/4/9/19/23/25 10/10/14/20/21/22/29/38        0.985
      A all15 kmeans 3              2/8/77                34/61/69        0.461
      A all15 kmeans 4           2/8/27/50             31/34/38/61        0.624
      A all15 kmeans 5         2/5/8/29/43          14/22/31/38/59        0.698
      A all15 kmeans 6      2/5/6/17/25/32       10/14/21/22/38/59        0.739
      A all15 kmeans 7    2/3/5/5/18/25/

## What the adjusted Rand index (ARI) says

**The modules largely do not break.** Cohort A reaches 0.80; cohort C's `union` hierarchical cut at
k = 4 reaches 0.886. Free clustering very nearly reproduces a partition it was never given.

**Cohort B's ARI is 0.000 everywhere, which is arithmetic, not a finding.** B's panel is a single
module, so the WGCNA label vector is constant, and the adjusted Rand index against a constant
partition is 0 by construction.

**Cohort A carries extreme samples.** A produces a 1-patient cluster at k = 4 and k = 5 and an
11/76 split at k = 2, in both arms. B and C do not. Step 15 follows this up.

## Where this was run

Rendered notebooks are committed, so each records the machine, the R and the
package versions that produced its output.


In [2]:
run_provenance()

run on   : Annes-MacBook-Pro-193.local ( Darwin 27.0.0 )
date     : 2026-09-25 12:05 EDT 
R        : R version 4.4.3 (2025-02-28) | x86_64-apple-darwin13.4.0 
R comes from: /Users/adeslatt/miniforge3/envs/endotypes-proteomics 
packages :
   WGCNA            1.74
   ComplexHeatmap   2.22.0
   sva              3.54.0
   VarSelLCM        2.1.3.2
   cluster          2.1.8.1
   fpc              2.2.15
   circlize         0.4.18
